# ROS2

Topic map, **`rgbd_pose_node.py`** (Python detector on RGB-D topics), TF, C++ latency monitor, and Docker demo (`./ros2/run_demo.sh`, `--viz` for RViz GIF).

In [1]:
import os
import runpy

_cwd = os.getcwd()
_utils = os.path.join(_cwd, "colab_utils.py")
if not os.path.isfile(_utils):
    _utils = os.path.join(_cwd, "notebooks", "colab_utils.py")
runpy.run_path(_utils)

from notebooks.colab_utils import setup_notebook

PROJECT_ROOT = setup_notebook()
ros2_dir = os.path.join(PROJECT_ROOT, "ros2", "object_pose")
with open(os.path.join(ros2_dir, "scripts", "rgbd_pose_node.py"), encoding="utf-8") as f:
    pose_node_src = f.read()
with open(os.path.join(ros2_dir, "src", "pose_latency_monitor.cpp"), encoding="utf-8") as f:
    cpp_src = f.read()

## Topics

| Stage | Topic | Type |
|-------|-------|------|
| RGB | `/rgb/image_raw` | `sensor_msgs/Image` |
| Depth | `/depth/image_raw` | `sensor_msgs/Image` (`16UC1` mm) |
| Intrinsics | `/camera/camera_info` | `sensor_msgs/CameraInfo` |
| Pose | `/object_pose` | `geometry_msgs/PoseStamped` |
| TF | `object_frame` | child of `camera_optical_frame` |

Pose is estimated in the **camera optical frame** (Z forward). Apply the standard optical↔`camera_link` rotation when bridging to nodes that expect the mount frame.

## Data flow

```
demo_rgbd_publisher.py
  /rgb/image_raw, /depth/image_raw, /camera/camera_info
        │
        ▼
rgbd_pose_node.py  (ObjectDetector + PoseEstimator)
        │
        ├──► /object_pose (PoseStamped)
        └──► TF object_frame → camera_optical_frame
        │
        ▼
pose_latency_monitor (C++)  ·  RViz (--viz)
```

**Run (Docker, repo root):**

```bash
./ros2/run_demo.sh              # headless latency logs
./ros2/run_demo.sh --viz        # record assets/demo/ros2_rviz_demo.gif
```

## Python pose node

`ros2/object_pose/scripts/rgbd_pose_node.py` syncs RGB-D + `CameraInfo`, runs the same detector/pose code as `pipeline.ipynb`, and publishes `/object_pose` + TF.

In [2]:
print(pose_node_src)

#!/usr/bin/env python3
# ros2/object_pose/scripts/rgbd_pose_node.py

import os
import sys

import numpy as np
import rclpy
from cv_bridge import CvBridge
from geometry_msgs.msg import PoseStamped, TransformStamped
from message_filters import ApproximateTimeSynchronizer, Subscriber
from rclpy.node import Node
from sensor_msgs.msg import CameraInfo, Image
from tf2_ros import TransformBroadcaster

_SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.environ.get("PROJECT_ROOT") or os.path.abspath(
    os.path.join(_SCRIPT_DIR, "..", "..", "..")
)
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from data.geometry import CameraIntrinsics
from rgbd_pose.detection import ObjectDetector
from rgbd_pose.pose import PoseEstimator


def camera_info_to_intrinsics(info):
    k = np.array(info.k, dtype=np.float64).reshape(3, 3)
    return CameraIntrinsics.from_matrix(k, int(info.width), int(info.height))


class RgbdPoseNode(Node):

    def __init__(self):
   

## C++ latency monitor

`pose_latency_monitor.cpp` subscribes to `/object_pose` and prints stamp latency plus rolling FPS.

In [3]:
print(cpp_src)

/**
 * ROS2 subscriber: monitors /object_pose latency and FPS.
 *
 *   ros2 run object_pose pose_latency_monitor
 *
 * Local demo (Docker, no system ROS2 install):
 *   ./ros2/run_demo.sh
 */

#include <chrono>
#include <cmath>
#include <iostream>
#include <string>

#include "geometry_msgs/msg/pose_stamped.hpp"
#include "rclcpp/rclcpp.hpp"

using namespace std::chrono_literals;

class PoseLatencyMonitor : public rclcpp::Node {
 public:
  PoseLatencyMonitor() : Node("pose_latency_monitor") {
    sub_ = create_subscription<geometry_msgs::msg::PoseStamped>(
        "/object_pose", rclcpp::SensorDataQoS(),
        [this](const geometry_msgs::msg::PoseStamped::SharedPtr msg) {
          this->on_pose(msg);
        });

    RCLCPP_INFO(get_logger(), "Listening on /object_pose");
  }

 private:
  void on_pose(const geometry_msgs::msg::PoseStamped::SharedPtr msg) {
    const auto now = this->now();
    const rclcpp::Time stamp(msg->header.stamp);

    double latency_ms = (now - stamp).seconds(